In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

In [4]:
import asyncio
from google import genai

client = genai.Client(api_key=api_key)

model = "gemini-2.0-flash-live-001"
config = {"response_modalities": ["TEXT"]}



In [5]:
async def connect():
    async with client.aio.live.connect(
        model=model,
        config=config,
    ) as session:
        print("Connected to Gemini Live session")
        
        

In [6]:
connect()

<coroutine object connect at 0x00000198E9C19000>

In [11]:
async def send_receive():
    async with client.aio.live.connect(model=model,config=config,) as session:
        message = " Hii, How are you?"
        print(f"Sending message: {message}")
        await session.send_client_content(
            turns={
                "role": "user",
                "parts": [
                    {
                        "text": message,
                    }
                ],
            },
            turn_complete=True,
        )
        
        async for response in session.receive():
            if response.text is not None:
                print(f"Received response: {response.text}")

In [13]:
import nest_asyncio
nest_asyncio.apply()
await send_receive()

Sending message:  Hii, How are you?
Received response: I am
Received response:  doing well, thank you for asking! How are you today?



In [14]:
import io
from pathlib import Path
from google import genai
from google.genai import types
import soundfile as sf
import librosa

In [16]:
async def send_and_receive_audio():
    async with client.aio.live.connect(model=model, config=config) as session:
        
        buffer = io.BytesIO()
        y, sr = librosa.load("C:\\Users\\SIDHYA\\Development\\Freelance\\Testing\\gemini-multimodal-live-api\\audio\\16000.wav", sr=16000)
        sf.write(buffer, y, sr, format='RAW', subtype='PCM_16')
        buffer.seek(0)
        audio_bytes = buffer.read()
        
        
        await session.send_realtime_input(
            audio=types.Blob(
                data=audio_bytes, mime_type="audio/pcm;rate=16000" 
                ),
        )
        
        async for response in session.receive():
            if response.text is not None:
                print(f"Received response: {response.text}")
                
                
nest_asyncio.apply()
await send_and_receive_audio()

c:\Users\SIDHYA\Development\Freelance\Testing\gemini-multimodal-live-api\.venv\Lib\site-packages\llvmlite\ir\values.py:860: RuntimeWarning: coroutine 'send_receive' was never awaited
  def __init__(self, args=()):


Received response: Yes
Received response: , I can hear you. How can I help you today?



In [24]:
import wave

config = {"response_modalities": ["AUDIO"]}

async def receive_audio():
    async with client.aio.live.connect(model=model, config=config) as session:
        wf = wave.open("audio.wav", "wb")
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(24000)

        message = "Who is narendra modi?"
        await session.send_client_content(
            turns={"role": "user", "parts": [{"text": message}]}, turn_complete=True
        )

        async for response in session.receive():
            
            if response.data is not None:
                wf.writeframes(response.data)

            # Un-comment this code to print audio data info
            # if response.server_content.model_turn is not None:
            #      print(response.server_content.model_turn.parts[0].inline_data.mime_type)

        wf.close()
        
        
nest_asyncio.apply()
await receive_audio()
